# MISRA C: zip in, fixes and report out

**Part 1 (steps 0-5):** upload a zip of your C project -> find MISRA C:2012 findings -> an AI proposes fixes ->
the checker keeps a fix only if the gates pass -> you download `report.html`, `findings.csv`, `fixed_project.zip`.
**Part 2 (steps 6-8, optional):** train your own model on the accepted fixes.

| Step | What | Run when |
|---|---|---|
| 0 | Settings (one form, all choices here) | every session |
| 1 | Install tools, get code, build | every session |
| 2 | Connect the AI and test it | every session |
| 3 | Upload your project zip | each project |
| 4 | Run: find, fix, report | each project |
| 5 | Show and download results | each project |
| 6-8 | Dataset, train, save the model | optional |

**Use Runtime > Run all** the first time, or run the steps top to bottom. After a disconnect, rerun 0-2.

The checker's gates prove "still compiles, targeted findings gone, nothing worse", not that behaviour is unchanged.
This is not a MISRA compliance claim. Review every patch.

## Step 0 - Settings
Change values in the form, then run the cell. Nothing else in the notebook needs editing.

In [ ]:
#@title Settings { run: "auto" }
REPO = "arrunraj66/misra-checker" #@param {type:"string"}
BRANCH = "claude/confident-dijkstra-n8hoso" #@param {type:"string"}
AI = "claude" #@param ["claude", "ollama-local-model", "offline-test"]
CLAUDE_MODEL = "claude-sonnet-5-5" #@param {type:"string"}
OLLAMA_MODEL = "qwen2.5-coder:7b" #@param {type:"string"}
YOUR_GGUF_ON_DRIVE = "" #@param {type:"string"}
C_STANDARD = "c99" #@param ["c90", "c99", "c11", "c17"]
DEFINES = "" #@param {type:"string"}
EXCLUDE_FOLDERS = "" #@param {type:"string"}
MAX_FILES = 0 #@param {type:"integer"}
VERIFY_COMMAND = "true" #@param {type:"string"}
FILE_TIMEOUT_MINUTES = 30 #@param {type:"integer"}

DEFINE_LIST = DEFINES.split()                                   # e.g. "STM32F4 DEBUG=1"
EXCLUDE_LIST = [g for g in EXCLUDE_FOLDERS.replace(",", " ").split() if g]   # e.g. "lib/* vendor/*"; empty = whole project
print(f"AI={AI}  branch={BRANCH}  files={'ALL' if MAX_FILES == 0 else MAX_FILES}  "
      f"excluded={EXCLUDE_LIST or 'nothing (whole project: lib, vendor, drivers included)'}")

## Step 1 - Install, get the code, build (about 5 minutes the first time)

In [ ]:
#@title Install, clone, build { display-mode: "form" }
import os, subprocess, sys

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise SystemExit(f"FAILED: {cmd}")
    return r.stdout

print("os:", sh("grep PRETTY /etc/os-release").strip())
sh("apt-get update -qq && apt-get install -y -qq clang-14 llvm-14-dev libclang-14-dev libclang-cpp14-dev cmake ninja-build zstd")
print("tools installed")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")      # only needed for a private repo
except Exception:
    token = None
url = f"https://{token + '@' if token else ''}github.com/{REPO}.git"
SRC = "/content/misra-checker"
if not os.path.isdir(SRC + "/.git"):
    sh(f"git clone {url} {SRC}")
sh(f"cd {SRC} && git fetch origin {BRANCH} && git checkout -f -B work FETCH_HEAD")     # always the chosen branch
print("code:", sh(f"cd {SRC} && git log --oneline -1").strip())
assert os.path.exists(SRC + "/scripts/run_project_pipeline.py"), \
    f"branch {BRANCH} has no scripts/run_project_pipeline.py - set BRANCH in step 0"

sh(f"cd {SRC} && cmake -S . -B build -G Ninja -DLLVM_DIR=/usr/lib/llvm-14/lib/cmake/llvm && cmake --build build")
print("rules registered:", sh(f"cd {SRC} && ./build/misra-checker --list-rules | wc -l").strip())

# Development licence (the build uses the development key). Replace for real use.
open("/content/dev.key", "w").write("development-key-do-not-ship")
lic = sh(f"cd {SRC} && python3 scripts/issue_license.py --customer colab --expires 2027-12-31 --features convert --key-file /content/dev.key")
open("/content/colab.lic", "w").write(lic)
print("setup OK")

## Step 2 - Connect the AI and test it
The cell picks the AI you chose in step 0 and sends one small test question, so a bad key or model fails here, not an hour into a run.

- **claude:** add a Colab secret `ANTHROPIC_API_KEY` (key icon, left bar) and switch on *Notebook access*. Your code is sent to Anthropic.
- **ollama-local-model:** needs *Runtime > Change runtime type > T4 GPU*. Code stays in Colab. Set `YOUR_GGUF_ON_DRIVE` to use a model you trained.
- **offline-test:** no model; only rewrites Rules 7.1-7.3. Use it to check the whole flow works.

In [ ]:
#@title Connect and test the AI { display-mode: "form" }
import os, subprocess, time
S = "/content/misra-checker/scripts/ai_providers/"

if AI == "claude":
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    os.environ["MISRA_AI_MODEL"] = CLAUDE_MODEL
    PROVIDER = f"python3 {S}claude_provider.py"

elif AI == "ollama-local-model":
    r = subprocess.run("nvidia-smi -L", shell=True, text=True, capture_output=True)
    print(r.stdout.strip() or "WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then rerun from step 1.")
    r = subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, text=True, capture_output=True)
    if r.returncode:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit("Ollama install failed (text above). Use AI = claude, or send me this text.")
    subprocess.Popen("nohup ollama serve > /content/ollama.log 2>&1 &", shell=True)
    for _ in range(30):
        if subprocess.run("curl -s localhost:11434", shell=True, capture_output=True).returncode == 0: break
        time.sleep(2)
    if YOUR_GGUF_ON_DRIVE:
        from google.colab import drive; drive.mount("/content/drive")
        open("/content/Modelfile", "w").write(f"FROM {YOUR_GGUF_ON_DRIVE}\nPARAMETER temperature 0\nPARAMETER num_ctx 16384\n")
        subprocess.run("ollama create misra-fixer -f /content/Modelfile", shell=True, check=True)
        os.environ["MISRA_AI_MODEL"] = "misra-fixer"
    else:
        subprocess.run(f"ollama pull {OLLAMA_MODEL}", shell=True, check=True)
        os.environ["MISRA_AI_MODEL"] = OLLAMA_MODEL
    os.environ["MISRA_AI_NUM_CTX"] = "16384"
    PROVIDER = f"python3 {S}ollama_provider.py"

else:
    PROVIDER = f"python3 {S}deterministic_provider.py"

if AI != "offline-test":
    r = subprocess.run(PROVIDER, shell=True, input="Reply with the single word OK.", text=True, capture_output=True, timeout=600)
    print("test reply:", r.stdout.strip()[:200] or r.stderr[-1500:])
    assert r.returncode == 0, "AI test failed (text above) - fix this before step 4"
print("AI ready:", AI)

## Step 3 - Upload your project zip
Zip the folder that holds `main.c`, `uart.c`, `usb.c`, the headers and the bundled libraries (`lib`, `vendor`, `drivers`).

In [ ]:
#@title Upload zip { display-mode: "form" }
from google.colab import files
up = files.upload()
name = next(iter(up))
ZIP = "/content/" + name
open(ZIP, "wb").write(up[name])
import zipfile
n = zipfile.ZipFile(ZIP).namelist()
print(ZIP, "-", sum(x.endswith(".c") for x in n), ".c files,", sum(x.endswith(".h") for x in n), ".h files")

## Step 4 - Find, fix, report
Progress prints per file. A whole project can take a long time (one AI call per file). For a first try set
`MAX_FILES` to 3 in step 0. If the session ends, rerun 0-2 and this step; results are only written when the run finishes.

In [ ]:
#@title Run the pipeline { display-mode: "form" }
import subprocess, os
cmd = ["python3", "-u", "scripts/run_project_pipeline.py", ZIP, "--out", "/content/result",
       "--checker", "build/misra-checker", "--provider-cmd", PROVIDER,
       "--license", "/content/colab.lic", "--verify-cmd", VERIFY_COMMAND,
       "--std", C_STANDARD, "--max-files", str(MAX_FILES), "--timeout", str(FILE_TIMEOUT_MINUTES * 60)]
for d in DEFINE_LIST: cmd += ["-D", d]
for e in EXCLUDE_LIST: cmd += ["--exclude", e]
p = subprocess.Popen(cmd, cwd="/content/misra-checker", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
p.wait()
assert p.returncode == 0, f"pipeline failed with exit code {p.returncode}"

## Step 5 - Show and download the results

In [ ]:
#@title Report and downloads { display-mode: "form" }
import IPython, shutil, json
s = json.load(open("/content/result/report.json"))["summary"]
print(json.dumps(s, indent=1))
IPython.display.display(IPython.display.HTML(open("/content/result/report.html").read()))
shutil.make_archive("/content/misra_patches", "zip", "/content/result", "converter")
from google.colab import files
for f in ["report.html", "findings.csv", "fixed_project.zip"]:
    files.download("/content/result/" + f)
files.download("/content/misra_patches.zip")

---
# Part 2 (optional) - Train your own model
Every accepted fix from step 4 is a verified example. Run steps 3-5 on many projects first (`MAX_FILES = 0`), delete patches you would not accept in review, then train.
A few hundred pairs is a start. Use a **T4 GPU** runtime. Unsloth's interface changes often; if step 7 fails, check its current docs.

In [ ]:
#@title Step 6 - Build the dataset { display-mode: "form" }
!python3 /content/misra-checker/scripts/build_training_data.py /content/result/converter/* --dataset /content/train.jsonl --eval /content/eval.jsonl
!wc -l /content/train.jsonl /content/eval.jsonl

In [ ]:
#@title Step 7 - Fine-tune (Unsloth, Qwen2.5-Coder-7B) { display-mode: "form" }
import os, json
assert os.path.exists("/content/eval.jsonl"), "eval.jsonl is only written once there are enough pairs - collect more (step 4) first"
!pip install -q unsloth
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-Coder-7B-Instruct", max_seq_length=8192, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def load(path):
    rows = [json.loads(l) for l in open(path)]
    return Dataset.from_list([{"text": tok.apply_chat_template(r["messages"], tokenize=False)} for r in rows])

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=load("/content/train.jsonl"),
    eval_dataset=load("/content/eval.jsonl"),
    args=SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=2e-4, logging_steps=5, eval_strategy="epoch",
        output_dir="out", dataset_text_field="text", max_seq_length=8192, fp16=True))
trainer.train()

In [ ]:
#@title Step 8 - Save the model to Google Drive as .gguf { display-mode: "form" }
from google.colab import drive
drive.mount("/content/drive")
model.save_pretrained_gguf("/content/drive/MyDrive/misra-fixer", tok, quantization_method="q4_k_m")
print("saved to My Drive/misra-fixer - download the .gguf to your PC")

**Use the trained model:** in step 0 set `AI = ollama-local-model` and `YOUR_GGUF_ON_DRIVE` to the `.gguf` path, then rerun steps 2-5 on files that were **not** in training.
Keep the fine-tuned model only if more of its fixes pass the gates than the base model's.
**On your PC (RX 6700 XT):** put the `.gguf` in `./models/`, run `docker compose --profile llm up -d`, then `docker/import-model.sh models/<file>.gguf misra-fixer` (see `docs/ai-training-guide.md`, section 6).